# 手术调度问题

**类别：** 调度

来源: [https://www.hexaly.com/templates/surgery-scheduling-problem](https://www.hexaly.com/templates/surgery-scheduling-problem)


## 问题

**在手术调度问题中**，我们考虑一家医院，其拥有固定数量的可用手术室以及一组待安排的手术。每台手术具有给定的处理时间和所需的护士数量。一台手术只有在手术室可用且有足够的护士时才可开始。此外，对护士的班次还有最早开始时间、最晚结束时间以及最长持续时间的约束。手术调度问题的目标是寻找一种手术顺序，使 makespan（即所有手术处理完成的时间）最小化。

	

### 学到的建模原则

- 添加 [interval 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模各手术
- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模手术到手术室和护士的分配以及它们的顺序
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 将 interval 变量和 list 变量联系起来


## 数据

数据文件的格式如下：

- 第一行：手术室数量、护士数量、手术数量
- 第二行：每台手术的最早开始时间（单位：小时）
- 第三行：每台手术的最晚结束时间（单位：小时）
- 第四行：每台手术的持续时间（单位：分钟）
- 第五行：每台手术所需的护士数量
- 第六行：每位护士班次的最早开始时间（单位：小时）
- 第七行：每位护士班次的最晚结束时间（单位：小时）
- 第八行：班次的最大持续时长（单位：小时）
- 接下来的若干行：

- 对每台手术以及每个手术室：若该手术室与该手术不兼容则为 1，否则为 0。


## 模型

手术调度问题的 Hexaly 模型使用两类决策变量：interval 与 list。interval 变量用于建模各手术的时间区间，而 list 变量用于表示每个手术室以及每位护士所安排的手术顺序。我们将每个 interval 的长度约束为相应手术的持续时间。

通过对表示手术室的 list 变量使用 [**partition（划分）**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition) 运算符，我们确保每台手术恰好被分配到一个手术室。

为确保同一手术室同一时刻不会进行多台手术，我们施加不重叠（no-overlap）约束：在某手术室中，前一台手术结束之前，任何手术都不能开始。为建模该约束，我们定义了一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来表达相邻两台手术之间的关系。然后将该函数在每个手术室中安排的所有手术上，通过可变参数数量的 **and（与）** 运算符组合使用。需要注意的是，这些 **and** 表达式中的项数在搜索过程中会随着 list 的大小（即每个手术室安排的手术数）变化而变化。

通过访问每位护士所对应的 list 变量的第一个和最后一个元素，我们可以计算其首台手术的开始时间和末台手术的结束时间，从而对其班次的最早开始、最晚结束以及最大持续时间施加约束。由于每位护士同一时刻只能参与一台手术，我们还为护士添加了不重叠约束。

每台手术都有最低护士数量要求。使用 **contains（包含）** 运算符，我们可以判断每位护士是否参与了某台手术。将该运算符的结果累加起来即可得到每台手术对应的护士数量。

目标是使 makespan（即所有手术中的最晚结束时间）最小化。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def read_instance(filename):
    with open(filename) as f:
        lines = f.readlines()

    first_line = lines[0].split()
    # Number of rooms
    num_rooms = int(first_line[0])
    # Number of nurses
    num_nurses = int(first_line[1])
    # Number of surgeries
    num_surgeries = int(first_line[2])

    # Minimum start of each surgery
    line_min_start = lines[1].split()
    min_start = [int(line_min_start[s]) * 60 for s in range(num_surgeries)]

    # Maximum end of each surgery
    line_max_end = lines[2].split()
    max_end = [int(line_max_end[s]) * 60 for s in range(num_surgeries)]

    # Duration of each surgery
    line_duration = lines[3].split()
    duration = [int(line_duration[s]) for s in range(num_surgeries)]

    # Number of nurses needed for each surgery
    line_nurse_needed = lines[4].split()
    needed_nurses = [int(line_nurse_needed[s]) for s in range(num_surgeries)]

    # Earliest starting shift for each nurse
    line_earliest_shift = lines[5].split()
    shift_earliest_start = [int(line_earliest_shift[s]) * 60 for s in range(num_nurses)]

    # Latest ending shift for each nurse
    line_latest_shift = lines[6].split()
    shift_latest_end = [int(line_latest_shift[s]) * 60 for s in range(num_nurses)]

    # Maximum duration of each nurse's shift
    max_shift_duration = int(lines[7].split()[0]) * 60
    
    #Incompatible rooms for each surgery
    incompatible_rooms = [[0 for r in range(num_rooms)] for s in range(num_surgeries)]
    for s in range(num_surgeries):
        line = lines[8+s].split()
        for r in range(num_rooms):
            incompatible_rooms[s][r] = int(line[r])

    return (num_rooms, num_nurses, num_surgeries, min_start, max_end, 
            needed_nurses, shift_earliest_start, shift_latest_end, 
            max_shift_duration, incompatible_rooms, duration)

def main(instance_file, output_file, time_limit):
    num_rooms, num_nurses, num_surgeries, min_start, max_end, needed_nurses, \
    shift_earliest_start, shift_latest_end, max_shift_duration, \
    incompatible_rooms, duration = read_instance(instance_file)
    
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Sequence of surgery for each room
        surgery_order = [model.list(num_surgeries) for _ in range(num_rooms)]
        rooms = model.array(surgery_order)

        # Each surgery is scheduled in a room
        model.constraint(model.partition(rooms))

        # Only compatible rooms can be selected for a surgery
        for s in range(num_surgeries):
            for r in incompatible_rooms[s]:
                model.constraint(model.contains(surgery_order[r], s) == 0)

        # For each surgery, the selected room
        # This variable is only used to export the solution
        selected_room = [model.find(rooms, s) for s in range(num_surgeries)]

        # Interval decisions: time range of each surgery
        # Each surgery cannot start before and end after a certain time
        surgeries = [model.interval(min_start[s], max_end[s]) for s in range(num_surgeries)]

        for s in range(num_surgeries):
            # Each surgery has a specific duration
            model.constraint(model.length(surgeries[s]) == duration[s])

        surgery_array = model.array(surgeries)

        # A room can only have one surgery at a time
        for r in range(num_rooms):
            sequence = surgery_order[r]
            sequence_lambda = model.lambda_function(
                lambda s: surgery_array[sequence[s]] < surgery_array[sequence[s + 1]])
            model.constraint(
                model.and_(model.range(0, model.count(sequence) - 1), sequence_lambda)
            )

        # Each surgery needs a specific amount of nurse
        nurse_order = [model.list(num_surgeries) for _ in range(num_nurses)]

        for n in range(num_nurses):
            # Each nurse has an earliest starting shift and latest ending shift to be respected
            sequence = nurse_order[n]
            first_surgery_start = model.iif( 
                model.count(sequence) > 0, 
                model.start(surgery_array[sequence[0]]),
                shift_earliest_start[n]
            )
            last_surgery_end = model.iif(
                model.count(sequence) > 0, 
                model.end(surgery_array[sequence[model.count(sequence)-1]]),
                shift_earliest_start[n]
            )
            
            model.constraint(first_surgery_start >= shift_earliest_start[n])
            model.constraint(last_surgery_end <= shift_latest_end[n])

            # Each nurse cannot work more than a certain amount of hours
            model.constraint(last_surgery_end - first_surgery_start <= max_shift_duration)
            
            # Each nurse can only be at one operation at a time and stays all along the surgery
            sequence_lambda = model.lambda_function(
                lambda s: surgery_array[sequence[s]] < surgery_array[sequence[s + 1]])
            model.constraint(model.and_(
                model.range(0, model.count(sequence) - 1), sequence_lambda))
        
        # Each surgery needs a certain amount of nurses 
        nurse_order_array = model.array(nurse_order)
        for s in range(num_surgeries):
            model.constraint(
                model.sum(
                    model.range(0, num_nurses), 
                    model.lambda_function(lambda n : model.contains(nurse_order_array[n], s))
                )
                >= needed_nurses[s]
            )

        # Minimize the makespan: end of the last task
        makespan = model.max([model.end(surgeries[s]) for s in range(num_surgeries)])
        model.minimize(makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        # Write the solution in a file with the following format:
        # - for each surgery, the selected room, the start and end dates, 
        # the nurses working on this operation
        if output_file != None:
            with open(output_file, "w") as f:
                print("Solution written in file", output_file)
                list_nurses = {}
                for n in range(num_nurses):
                    surg_nurse = nurse_order[n].value
                    for s in surg_nurse:
                        if s not in list_nurses:
                            list_nurses[s] = [n]
                        else:
                            list_nurses[s].append(n)
                for s in range(num_surgeries):
                    f.write(str(s) + "\t"
                        + "\t" + str(selected_room[s].value)
                        + "\t" + str(surgeries[s].value.start())
                        + "\t" + str(surgeries[s].value.end()) 
                        + "\t" + str(list_nurses[s]) + "\n")



if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python surgeries_scheduling.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20
    main(instance_file, output_file, time_limit)
